# Notebook 11 — Nested Three-Model Ensemble & Component Ablation

**Research question:** Does a combined *descriptor LightGBM + fine-tuned CheMeleon + cross-fitted screening LightGBM* outperform its components under **honest, scaffold-grouped, nested model selection**? Which component has an independent contribution for each CYP isoform?

The three components are:

1. **Descriptor LGBM:** Morgan count fingerprints + RDKit descriptors, per CYP.
2. **CheMeleon fine-tuned:** 4-output Chemprop graph model initialized from the released, pretrained CheMeleon message-passing weights.
3. **ScreenStack LGBM:** same molecular features **plus four structure-predicted, cross-fitted screening responses**; never measured screen values at test time.

### Why this is genuinely nested

For each outer scaffold fold, separate inner scaffold folds create out-of-fold predictions from **all three retrained components**. We select nonnegative simplex weights from only those inner predictions. We then train the three components on the outer-training compounds and evaluate those **fixed selected weights** on the completely untouched outer fold. The ablation repeats the same process for each singleton, pair, and the triple. No outer-fold label selects weights.

**Cost:** This requires `(inner_folds + 1) × outer_folds` CheMeleon fine-tunings, as well as multiple LightGBM fits. Start in smoke mode. A full five-fold run with three inner folds requires **20 graph fine-tunings**, plus screening cross-fits; use Colab GPU and persistent Drive caching.

**Caveats:** The ordinary MAE here is *not* the challenge's official MA-ST-RAE. The notebook also reports a descriptive distance outside reported confidence intervals, but does not call it the official score. Any externally pre-trained model has its own pretraining-data provenance; this experiment only guards against leakage from the OpenADMET assay labels.

## 0 · Install dependencies
Run this cell in Colab. The synthetic smoke-test harness can skip installation if dependencies are preinstalled.

In [ ]:
%pip -q install "rdkit>=2023.9" "lightgbm>=4.3" "scikit-learn>=1.6" "scipy>=1.11" "pandas>=2" "matplotlib>=3.7" "chemprop>=2.2.0,<2.4" "lightning>=2.4"

## 1 · Configuration and reproducibility

A **proxy graph model is provided solely for testing notebook mechanics** when a real CheMeleon checkpoint cannot be downloaded. It is **not CheMeleon**. For any scientific run, `GRAPH_PROXY_SMOKE_ONLY` **must be `False`**; the notebook marks proxy results invalid for scientific comparison.

If using Drive, mount before executing this cell and set `PERSISTENT_ROOT` to a new experiment directory. Changing model settings changes the cache signature automatically. Existing Notebook 05/06/08 files may be used for **fold verification only**, never for nested weight selection.

In [ ]:
from pathlib import Path
from urllib.request import urlopen, urlretrieve
from itertools import combinations, product
import os, io, sys, json, time, zipfile, hashlib, platform, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

SEED=2026
FAST_MODE=True  # 1 outer fold, 2 inner folds, short training; change to False for full research run.
N_OUTER_FOLDS=5
INNER_FOLDS=2 if FAST_MODE else 3
SCREEN_CROSSFIT_FOLDS=2 if FAST_MODE else 3
FOLDS_TO_RUN=[0] if FAST_MODE else list(range(N_OUTER_FOLDS))
GRAPH_PROXY_SMOKE_ONLY=False  # FOR SYNTHETIC/OFFLINE EXECUTION TESTS ONLY; invalidates scientific results.
RUN_GRAPH=True
MAX_GRAPH_EPOCHS=3 if FAST_MODE else 22
GRAPH_PATIENCE=2 if FAST_MODE else 5
GRAPH_BATCH_SIZE=64
N_LGBM_ESTIMATORS=65 if FAST_MODE else 350
N_SCREEN_ESTIMATORS=55 if FAST_MODE else 180
FINGERPRINT_BITS=2048
WEIGHT_STEP=0.25  # Convex weights; pre-register and do NOT tune on outer results.
DATA_OVERRIDES={}  # {'direct':'/content/direct.csv','single':'/content/screen.csv'} for controlled tests.
PERSISTENT_ROOT=None  # e.g. '/content/drive/MyDrive/CYP/Notebook11'
CHECK_FOLD_ZIPS=[]  # optional Notebook05/06/08 results for fold manifest verification only.
EXPERIMENT_TAG='nested_v2'  # increment when changing methodological code.
EXPORT_ZIP='cyp_notebook_11_results.zip'
ISOFORMS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
MODELS=['Descriptor_LGBM','CheMeleon_finetuned','ScreenStack_LGBM']
BASE=Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')
OUT=BASE/'cyp_notebook_11_results';OUT.mkdir(parents=True,exist_ok=True)
CACHE=BASE/'cyp_notebook_11_cache';CACHE.mkdir(parents=True,exist_ok=True)
ROOT_URL='https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/'
FILES={'direct':'cyp-challenge-TRAIN_inhibition.csv','single':'cyp-challenge-single-concentration-TRAIN.csv'}
random.seed(SEED);np.random.seed(SEED)
print('Run mode:', 'FAST smoke test' if FAST_MODE else 'FULL five-fold nested CV')
print('Python:',platform.python_version(),'RDKit:',rdBase.rdkitVersion,'LightGBM:',lgb.__version__)
if GRAPH_PROXY_SMOKE_ONLY:
    print('WARNING: proxy graph smoke test! Outputs are NOT SCIENTIFIC RESULTS.')

## 2 · Load direct inhibition and single-concentration screening data

The released screening table uses `enzyme` and `log2fc_estimate`. All joins use canonical isomeric SMILES and reject duplicate direct-training identities. We reproduce the Notebook 05 outer GroupKFold assignments; all auxiliary-only compounds inherit their chemical scaffold group.

In [ ]:
def download_source(key):
    if key in DATA_OVERRIDES: blob=Path(DATA_OVERRIDES[key]).read_bytes()
    else:
        with urlopen(ROOT_URL+FILES[key],timeout=240) as response:blob=response.read()
    return pd.read_csv(io.BytesIO(blob)),hashlib.sha256(blob).hexdigest()

source={};data_sha={}
for key in FILES:
    source[key],data_sha[key]=download_source(key)
    print(key,source[key].shape,'SHA256:',data_sha[key][:18])

def canonical_scaffold(s):
    if not isinstance(s,str) or not s.strip():return None,None
    m=Chem.MolFromSmiles(s)
    if m is None:return None,None
    can=Chem.MolToSmiles(m,canonical=True,isomericSmiles=True)
    core=MurckoScaffold.GetScaffoldForMol(m)
    scaffold=(Chem.MolToSmiles(core,canonical=True,isomericSmiles=False)
              if core.GetNumAtoms() else 'ACYCLIC_'+can)
    return can,scaffold

primary=source['direct'].copy().reset_index(drop=True)
if 'SMILES' not in primary: raise KeyError('Direct table missing SMILES')
ident=[canonical_scaffold(s) for s in primary.SMILES]
primary['canonical_smiles']=[k for k,g in ident]
primary['scaffold_group']=[g for k,g in ident]
assert primary.canonical_smiles.notna().all(), 'Invalid structure in direct table'
assert primary.canonical_smiles.is_unique,'Direct data contain duplicate canonical SMILES: resolve before CV'
for iso in ISOFORMS: assert f'{iso}_pIC50_direct_inhibition' in primary,iso

fold_ids=np.full(len(primary),-1,int)
for f,(_,vi) in enumerate(GroupKFold(n_splits=N_OUTER_FOLDS).split(primary,groups=primary.scaffold_group)):
    fold_ids[vi]=f
primary['fold']=fold_ids
manifest=primary[['canonical_smiles','scaffold_group','fold']].copy()
assert (fold_ids>=0).all() and manifest.groupby('scaffold_group').fold.nunique().max()==1
manifest.to_csv(OUT/'fold_manifest.csv',index=False)

for archive in CHECK_FOLD_ZIPS:
    with zipfile.ZipFile(archive) as z:
        path=[x for x in z.namelist() if x=='fold_manifest.csv' or x.endswith('/fold_manifest.csv')]
        assert len(path)==1, f'Expected one fold manifest in {archive}'
        old=pd.read_csv(io.BytesIO(z.read(path[0])))
    a=manifest.set_index('canonical_smiles').fold
    b=old.set_index('canonical_smiles').fold
    assert b.index.is_unique and set(a.index)==set(b.index)
    assert np.array_equal(a.to_numpy(),b.loc[a.index].to_numpy()),'Fold mismatch against '+archive
    print('Verified previous fold manifest:',archive)

# Deduplicate screening repeated compound-CYP measurements by mean response.
screen=source['single'].copy()
for req in ['SMILES','enzyme','log2fc_estimate']:
    assert req in screen, f'Screening table missing {req}: {list(screen.columns)}'
screen['canonical_smiles']=[canonical_scaffold(s)[0] for s in screen.SMILES]
screen['screen_value']=pd.to_numeric(screen.log2fc_estimate,errors='coerce')
screen['enzyme']=screen.enzyme.astype(str).str.strip().str.upper()
screen=screen.dropna(subset=['canonical_smiles','screen_value'])
screen=screen[screen.enzyme.isin(ISOFORMS)]
aggregated=screen.groupby(['canonical_smiles','enzyme'],as_index=False).screen_value.mean()

all_smiles=list(primary.canonical_smiles)
extra=sorted(set(aggregated.canonical_smiles)-set(all_smiles))
all_smiles+=extra
N_PRIMARY=len(primary);N_ALL=len(all_smiles)
all_index={s:i for i,s in enumerate(all_smiles)}
all_groups=np.asarray([canonical_scaffold(s)[1] for s in all_smiles],object)
Y=np.column_stack([pd.to_numeric(primary[f'{iso}_pIC50_direct_inhibition'],errors='coerce').to_numpy(float) for iso in ISOFORMS])
SCREEN_Y=np.full((N_ALL,4),np.nan,float)
for row in aggregated.itertuples(index=False):
    SCREEN_Y[all_index[row.canonical_smiles],ISOFORMS.index(row.enzyme)]=row.screen_value
CI_LOW=np.column_stack([pd.to_numeric(primary.get(f'{iso}_pIC50_direct_inhibition_conf_low',pd.Series(np.nan,index=primary.index)),errors='coerce').to_numpy(float) for iso in ISOFORMS])
CI_HIGH=np.column_stack([pd.to_numeric(primary.get(f'{iso}_pIC50_direct_inhibition_conf_high',pd.Series(np.nan,index=primary.index)),errors='coerce').to_numpy(float) for iso in ISOFORMS])
assert len(np.unique(all_smiles))==N_ALL
assert (np.isfinite(SCREEN_Y).sum(axis=0)>0).all()
print('Primary molecules:',N_PRIMARY,'Auxiliary-only:',len(extra),'screen counts:',np.isfinite(SCREEN_Y).sum(axis=0),'direct counts:',np.isfinite(Y).sum(axis=0))

## 3 · Featurize Morgan counts and RDKit 2D descriptors once

RDKit molecular descriptors use no assay labels. Missing descriptors are median-imputed **within each model's training subset**; none of the held-out observations are used to learn preprocessing.

In [ ]:
feature_id=hashlib.sha256(json.dumps({'sha':data_sha,'bits':FINGERPRINT_BITS,
    'rdkit':rdBase.rdkitVersion,'desc':[n for n,_ in Descriptors._descList]},sort_keys=True).encode()).hexdigest()[:16]
feature_file=CACHE/f'features_{feature_id}.npz'
if feature_file.exists():
    with np.load(feature_file) as z: X_RAW=z['X_RAW']
    print('Loaded cached descriptors/fingerprints:',X_RAW.shape)
else:
    fg=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=FINGERPRINT_BITS,countSimulation=False)
    desc_funcs=[fn for _,fn in Descriptors._descList]
    X_RAW=np.empty((N_ALL,FINGERPRINT_BITS+len(desc_funcs)),np.float32)
    for i,smi in enumerate(all_smiles):
        mol=Chem.MolFromSmiles(smi)
        X_RAW[i,:FINGERPRINT_BITS]=np.asarray(fg.GetCountFingerprintAsNumPy(mol),np.float32)
        for j,fn in enumerate(desc_funcs):
            try:X_RAW[i,FINGERPRINT_BITS+j]=float(fn(mol))
            except Exception:X_RAW[i,FINGERPRINT_BITS+j]=np.nan
        if (i+1)%1000==0:print('Featurized',i+1,'/',N_ALL)
    X_RAW[~np.isfinite(X_RAW)]=np.nan
    np.savez_compressed(feature_file,X_RAW=X_RAW)
assert X_RAW.shape[0]==N_ALL
print('Feature matrix:',X_RAW.shape,'missing:',np.isnan(X_RAW).sum())

## 4 · Model implementations and strict leakage guardrails

The screen stack is **cross-fitted within every inner/outer model fit**. Stage-one screen regressors exclude all molecules sharing a scaffold with their own screening validation fold, the current inner validation fold, and the outer test fold. Stage-two models never receive observed screening measurements at inference time.

For CheMeleon, the Zenodo message-passing weights are loaded with `weights_only=True`. Local Lightning checkpoints created by this run may require `weights_only=False` for inference; do not load arbitrary external checkpoints unrestricted.

In [ ]:
import torch
from sklearn.model_selection import GroupKFold as _GroupKFold
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

torch.manual_seed(SEED);pl.seed_everything(SEED,workers=True)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:',torch.__version__,'GPU available:',torch.cuda.is_available())

if RUN_GRAPH and not GRAPH_PROXY_SMOKE_ONLY:
    from chemprop import featurizers, nn as cnn, data as cdata, models as cmodels
    import chemprop
    ZENODO='https://zenodo.org/records/15460715/files/chemeleon_mp.pt'
    CKPT=CACHE/'chemeleon_mp.pt'
    if not CKPT.exists():
        print('Downloading original CheMeleon message-passing weights from Zenodo')
        urlretrieve(ZENODO,CKPT)
    assert CKPT.stat().st_size>100000,'Unexpectedly small CheMeleon checkpoint'
    CKPT_SHA=hashlib.sha256(CKPT.read_bytes()).hexdigest()
    print('Pretrained checkpoint SHA256:',CKPT_SHA)
else:
    CKPT_SHA='PROXY_ONLY' if GRAPH_PROXY_SMOKE_ONLY else 'DISABLED'

SETTINGS={'tag':EXPERIMENT_TAG,'data_sha':data_sha,'folds':N_OUTER_FOLDS,
    'inner_folds':INNER_FOLDS,'screen_crossfit':SCREEN_CROSSFIT_FOLDS,
    'seed':SEED,'fingerprint_bits':FINGERPRINT_BITS,'n_lgbm':N_LGBM_ESTIMATORS,
    'n_screen':N_SCREEN_ESTIMATORS,'fast':FAST_MODE,'graph':RUN_GRAPH,
    'graph_proxy_test_only':GRAPH_PROXY_SMOKE_ONLY,'checkpoint_sha':CKPT_SHA,
    'graph_max_epochs':MAX_GRAPH_EPOCHS,'graph_patience':GRAPH_PATIENCE,
    'weight_step':WEIGHT_STEP,'lightgbm':lgb.__version__,'rdkit':rdBase.rdkitVersion,
    'python':platform.python_version(), 'torch':torch.__version__}
SIGNATURE=hashlib.sha256(json.dumps(SETTINGS,sort_keys=True).encode()).hexdigest()[:20]
RUN_CACHE=CACHE/SIGNATURE;RUN_CACHE.mkdir(parents=True,exist_ok=True)
print('Cache signature:',SIGNATURE,'destination:',RUN_CACHE)

# Every train/evaluation call is supplied the outer holdout groups; the code independently
# excludes inner evaluation groups. All IDs are canonical molecules, not assay rows.
AUDIT_LOG=[]
def guard_groups(train_idx,predict_idx,outer_holdout_idx,what):
    train_groups=set(all_groups[np.asarray(train_idx,dtype=int)])
    held_groups=set(all_groups[np.asarray(outer_holdout_idx,dtype=int)])
    pred_groups=set(all_groups[np.asarray(predict_idx,dtype=int)])
    overlap_outer=len(train_groups&held_groups)
    overlap_pred=len(train_groups&pred_groups)
    assert overlap_outer==0 and overlap_pred==0, f'Leakage in {what}: outer={overlap_outer}, validation={overlap_pred}'
    AUDIT_LOG.append(dict(step=what,n_fit=len(train_idx),n_pred=len(predict_idx),
        outer_scaffold_overlap=overlap_outer,validation_scaffold_overlap=overlap_pred))

def impute_for(train_idx,pred_idx):
    imp=SimpleImputer(strategy='median',keep_empty_features=True)
    fit=imp.fit_transform(X_RAW[train_idx]).astype(np.float32)
    val=imp.transform(X_RAW[pred_idx]).astype(np.float32)
    assert fit.shape[1]==val.shape[1]
    return fit,val

def named_fit_predict(model,x_fit,y_fit,x_eval):
    # LightGBM's sklearn wrapper retains DataFrame feature names; provide matching names at prediction.
    cols=[f'f_{k:05d}' for k in range(x_fit.shape[1])]
    model.fit(pd.DataFrame(x_fit,columns=cols),y_fit)
    return model.predict(pd.DataFrame(x_eval,columns=cols))

def lgbm_model(seed,stage='potency'):
    return lgb.LGBMRegressor(n_estimators=N_SCREEN_ESTIMATORS if stage=='screen' else N_LGBM_ESTIMATORS,
        learning_rate=.04,num_leaves=15,min_child_samples=6 if FAST_MODE else 15,
        colsample_bytree=.8,subsample=.85,subsample_freq=1,
        reg_lambda=3.,verbosity=-1,n_jobs=2,random_state=seed)

def fit_direct(train_idx,pred_idx,outer_idx,seed):
    guard_groups(train_idx,pred_idx,outer_idx,'direct_LGBM')
    a,b=impute_for(train_idx,pred_idx)
    result=np.full((len(pred_idx),4),np.nan,float)
    for j in range(4):
        selected=np.isfinite(Y[train_idx,j])
        if selected.sum()<6: raise ValueError('Insufficient direct labels for '+ISOFORMS[j])
        if np.std(Y[train_idx[selected],j])<1e-12:
            result[:,j]=float(np.mean(Y[train_idx[selected],j]));continue
        reg=lgbm_model(seed+j)
        result[:,j]=named_fit_predict(reg,a[selected],Y[train_idx[selected],j],b)
    return result

def predict_screen_from_fit(screen_fit_idx,query_idx,seed,why,outer_idx):
    guard_groups(screen_fit_idx,query_idx,outer_idx,why)
    x_fit,x_q=impute_for(screen_fit_idx,query_idx)
    result=np.full((len(query_idx),4),np.nan,float)
    for j in range(4):
        observed=np.isfinite(SCREEN_Y[screen_fit_idx,j])
        if observed.sum()<6 or np.std(SCREEN_Y[screen_fit_idx[observed],j])<1e-10:
            val=SCREEN_Y[screen_fit_idx[observed],j]
            result[:,j]=float(np.mean(val)) if len(val) else 0.;continue
        model=lgbm_model(seed+j,stage='screen')
        result[:,j]=named_fit_predict(model,x_fit[observed],SCREEN_Y[screen_fit_idx[observed],j],x_q)
    assert np.isfinite(result).all()
    return result

def fit_screenstack(train_idx,pred_idx,outer_idx,seed):
    # train_idx and pred_idx contain PRIMARY row indices; aux-only structures may be
    # admitted to FIRST-STAGE screen training iff their scaffold is eligible.
    guard_groups(train_idx,pred_idx,outer_idx,'screen_stack_second_stage')
    forbidden=set(all_groups[outer_idx])|set(all_groups[pred_idx])
    eligible=np.asarray([i for i,g in enumerate(all_groups) if g not in forbidden],int)
    t_groups=all_groups[train_idx]
    n_inner=min(SCREEN_CROSSFIT_FOLDS,len(set(t_groups)))
    if n_inner<2: raise ValueError('Need >=2 distinct training scaffolds for cross-fitted screening features')
    train_screen=np.full((len(train_idx),4),np.nan,float)
    for sf,(fit_pos,val_pos) in enumerate(_GroupKFold(n_splits=n_inner).split(train_idx,groups=t_groups)):
        cf_valid=train_idx[val_pos]
        val_groups=set(all_groups[cf_valid])
        sf_fit=np.asarray([k for k in eligible if all_groups[k] not in val_groups],int)
        train_screen[val_pos]=predict_screen_from_fit(sf_fit,cf_valid,seed+100*sf,
            f'screen_stage1_crossfit_{sf}',outer_idx)
    assert np.isfinite(train_screen).all()
    pred_screen=predict_screen_from_fit(eligible,pred_idx,seed+99999,
        'screen_stage1_eval',outer_idx)
    x_tr,x_eval=impute_for(train_idx,pred_idx)
    x_tr=np.column_stack([x_tr,train_screen]).astype(np.float32)
    x_eval=np.column_stack([x_eval,pred_screen]).astype(np.float32)
    result=np.full((len(pred_idx),4),np.nan,float)
    for j in range(4):
        observed=np.isfinite(Y[train_idx,j])
        if observed.sum()<6:raise ValueError('Too few direct labels for screen stack '+ISOFORMS[j])
        reg=lgbm_model(seed+2000+j)
        result[:,j]=named_fit_predict(reg,x_tr[observed],Y[train_idx[observed],j],x_eval)
    return result

def train_chemeleon(train_idx,pred_idx,outer_idx,seed,workdir):
    guard_groups(train_idx,pred_idx,outer_idx,'chemeleon_training')
    if GRAPH_PROXY_SMOKE_ONLY:
        # Proves folds/weight selection work without a checkpoint; NOT CheMeleon.
        pred=fit_direct(train_idx,pred_idx,outer_idx,seed+777)
        return pred
    if not RUN_GRAPH: raise RuntimeError('RUN_GRAPH must be True for three-model experiment')
    if DEVICE!='cuda':warnings.warn('Fine-tuning CheMeleon without GPU may be very slow')
    unique_groups=len(set(all_groups[train_idx]))
    if unique_groups<3:raise ValueError('Insufficient scaffolds for internal graph dev split')
    fit_pos,dev_pos=next(GroupShuffleSplit(n_splits=1,test_size=.15,random_state=seed).split(
        train_idx,groups=all_groups[train_idx]))
    fit_idx=train_idx[fit_pos];dev_idx=train_idx[dev_pos]
    guard_groups(fit_idx,dev_idx,outer_idx,'chemeleon_earlystop_dev')
    def point(i,labels):
        return cdata.MoleculeDatapoint.from_smi(all_smiles[i],np.asarray(labels,dtype=np.float32))
    featurizer=featurizers.SimpleMoleculeMolGraphFeaturizer()
    ds_train=cdata.MoleculeDataset([point(i,Y[i]) for i in fit_idx],featurizer)
    scaler=ds_train.normalize_targets()  # fit scaler only on fitting molecules
    ds_val=cdata.MoleculeDataset([point(i,Y[i]) for i in dev_idx],featurizer)
    ds_val.normalize_targets(scaler)
    ds_test=cdata.MoleculeDataset([cdata.MoleculeDatapoint.from_smi(all_smiles[i]) for i in pred_idx],featurizer)
    loaders=[cdata.build_dataloader(ds,batch_size=GRAPH_BATCH_SIZE,num_workers=0,shuffle=shuffle)
             for ds,shuffle in [(ds_train,True),(ds_val,False),(ds_test,False)]]
    checkpoint_obj=torch.load(CKPT,map_location='cpu',weights_only=True)
    mp=cnn.BondMessagePassing(**checkpoint_obj['hyper_parameters'])
    mp.load_state_dict(checkpoint_obj['state_dict'])
    predictor=cnn.RegressionFFN(n_tasks=4,input_dim=mp.output_dim,
        output_transform=cnn.UnscaleTransform.from_standard_scaler(scaler))
    net=cmodels.MPNN(mp,cnn.MeanAggregation(),predictor,batch_norm=False,
        metrics=[cnn.metrics.MAE()])
    ckpt_dir=Path(workdir)/'trusted_local_checkpoints';ckpt_dir.mkdir(parents=True,exist_ok=True)
    checkpoint=ModelCheckpoint(dirpath=ckpt_dir,filename='best',monitor='val_loss',mode='min',save_top_k=1)
    stopper=EarlyStopping(monitor='val_loss',mode='min',patience=GRAPH_PATIENCE)
    trainer=pl.Trainer(max_epochs=MAX_GRAPH_EPOCHS,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',devices=1,
        logger=False,enable_model_summary=False,enable_progress_bar=False,
        callbacks=[checkpoint,stopper],gradient_clip_val=1.0,log_every_n_steps=10)
    torch.manual_seed(seed)
    trainer.fit(net,loaders[0],loaders[1])
    assert checkpoint.best_model_path,'CheMeleon training produced no best checkpoint'
    with torch.inference_mode():
        batches=trainer.predict(net,dataloaders=loaders[2],ckpt_path='best',weights_only=False)
    pred=np.concatenate([np.asarray(t) for t in batches],axis=0).reshape(len(pred_idx),4).astype(float)
    del trainer,net,mp
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return pred

## 5 · Cached split predictions

Each individual `(outer fold, inner fold, model)` prediction is saved atomically with an exact index list and run signature. Cached models are never reused across altered data, settings, or checkpoint hashes. **One completed fold can be resumed without refitting its saved components.**

In [ ]:
CACHED_CALLS=[]
def predict_model(name,train_idx,pred_idx,outer_idx,fold,section,seed):
    path=RUN_CACHE/f'fold{fold}'/section/name
    path.mkdir(parents=True,exist_ok=True)
    npz=path/'predictions.npz'
    train_idx=np.asarray(train_idx,int);pred_idx=np.asarray(pred_idx,int)
    fingerprint=hashlib.sha256((SIGNATURE+name+str(seed)+
        train_idx.tobytes().hex()+pred_idx.tobytes().hex()+np.asarray(outer_idx,int).tobytes().hex()).encode()).hexdigest()
    audit_start=len(AUDIT_LOG)
    if npz.exists():
        with np.load(npz) as z:
            pred=z['pred']; actual_fingerprint=str(z['fingerprint'].item())
            old_idx=z['pred_idx']
            if 'audit_json' not in z.files:
                raise RuntimeError('Cached model has no persisted leakage audit; change EXPERIMENT_TAG')
            saved_audit=json.loads(str(z['audit_json'].item()))
        if actual_fingerprint==fingerprint and np.array_equal(old_idx,pred_idx):
            assert pred.shape==(len(pred_idx),4) and np.isfinite(pred).all(),f'Corrupt cache {npz}'
            assert saved_audit and all(r['outer_scaffold_overlap']==0 and r['validation_scaffold_overlap']==0 for r in saved_audit)
            # Also verify this split itself with the current compound->scaffold mapping.
            guard_groups(train_idx,pred_idx,outer_idx,f'cached_{name}')
            AUDIT_LOG.extend(saved_audit)
            CACHED_CALLS.append(dict(model=name,fold=fold,section=section,cached=True))
            print('  resumed',f'fold {fold}',section,name)
            return pred
        raise RuntimeError('Stale/mismatched cache; change EXPERIMENT_TAG or clear '+str(npz))
    if name=='Descriptor_LGBM':pred=fit_direct(train_idx,pred_idx,outer_idx,seed)
    elif name=='ScreenStack_LGBM':pred=fit_screenstack(train_idx,pred_idx,outer_idx,seed)
    elif name=='CheMeleon_finetuned':pred=train_chemeleon(train_idx,pred_idx,outer_idx,seed,path)
    else:raise ValueError(name)
    assert pred.shape==(len(pred_idx),4) and np.isfinite(pred).all()
    tmp=path/'predictions.tmp.npz'
    model_audit=AUDIT_LOG[audit_start:]
    assert model_audit and all(r['outer_scaffold_overlap']==0 and r['validation_scaffold_overlap']==0 for r in model_audit)
    np.savez_compressed(tmp,pred=pred,pred_idx=pred_idx,fingerprint=fingerprint,
                        audit_json=json.dumps(model_audit))
    os.replace(tmp,npz)
    CACHED_CALLS.append(dict(model=name,fold=fold,section=section,cached=False))
    print('  finished',f'fold {fold}',section,name)
    return pred

# Seven candidate families: 3 standalone, 3 pairs, 1 triple.
SUBSETS=[tuple(c) for size in [1,2,3] for c in combinations(MODELS,size)]

def blend_name(subset):return ' + '.join(subset)
def weight_grid(n,step=WEIGHT_STEP):
    q=round(1.0/step)
    assert abs(q*step-1.0)<1e-10
    for v in product(range(q+1),repeat=n):
        if sum(v)==q:yield np.asarray(v,float)/q

def macro_mae(yt,pred):
    yt=np.asarray(yt,float);pred=np.asarray(pred,float)
    values=[]
    for j in range(4):
        mask=np.isfinite(yt[:,j])&np.isfinite(pred[:,j])
        if mask.sum():values.append(mean_absolute_error(yt[mask,j],pred[mask,j]))
    return float(np.mean(values)) if values else np.nan

def choose_weights(inner_truth,inner_preds,subset):
    best=None
    for w in weight_grid(len(subset)):
        p=sum(float(x)*inner_preds[name] for name,x in zip(subset,w))
        score=macro_mae(inner_truth,p)
        if not np.isfinite(score):continue
        if best is None or score<best[0]-1e-12:best=(score,w)
    if best is None:raise ValueError('No admissible blend weights')
    return best

## 6 · Run **inner selection** and **outer evaluation**

Inner predictions are **training-only OOF** for weight selection; outer models are retrained separately. Per-isoform reports are evaluated without independently tuning isoform-specific weights. The outer fold is not used for any tuning, feature screening, early stopping, or sample weighting.

The current scratch/test mode can run one fold; the publication-style experiment requires all five folds **and** the actual CheMeleon checkpoint.

In [ ]:
selection_records=[];outer_records=[];inner_records=[]
for outer_fold in FOLDS_TO_RUN:
    outer_train=np.flatnonzero(fold_ids!=outer_fold)
    outer_test=np.flatnonzero(fold_ids==outer_fold)
    outer_groups=set(all_groups[outer_test])
    assert not outer_groups.intersection(set(all_groups[outer_train]))
    inner_groups=all_groups[outer_train]
    n_inner=min(INNER_FOLDS,len(set(inner_groups)))
    if n_inner<2: raise ValueError('Not enough distinct outer training scaffolds')
    inner_splitter=GroupKFold(n_splits=n_inner)
    # Every inner molecule receives one OOF prediction from each component.
    inner_predictions={model:np.full((len(outer_train),4),np.nan) for model in MODELS}
    for inner_f,(trpos,vpos) in enumerate(inner_splitter.split(outer_train,groups=inner_groups)):
        fit_idx=outer_train[trpos];val_idx=outer_train[vpos]
        assert not set(all_groups[fit_idx]).intersection(set(all_groups[val_idx]))
        for model in MODELS:
            pred=predict_model(model,fit_idx,val_idx,outer_test,outer_fold,f'inner{inner_f}',
                               SEED+outer_fold*10000+inner_f*100)
            inner_predictions[model][vpos]=pred
        for k,i in enumerate(val_idx):
            for j,iso in enumerate(ISOFORMS):
                if not np.isfinite(Y[i,j]):continue
                r=dict(fold=outer_fold,inner_fold=inner_f,canonical_smiles=all_smiles[i],isoform=iso,y_true=float(Y[i,j]))
                r.update({model:float(inner_predictions[model][vpos[k],j]) for model in MODELS})
                inner_records.append(r)
    for name,pred in inner_predictions.items():
        assert np.isfinite(pred).all(),f'Missing inner OOF predictions for {name}, outer={outer_fold}'
    # Learn seven different weight vectors, each using ONLY the inner OOF predictions.
    selected={}
    for subset in SUBSETS:
        score,w=choose_weights(Y[outer_train],inner_predictions,subset)
        key=blend_name(subset)
        selected[key]=(subset,w)
        selection_records.append(dict(fold=outer_fold,subset=key,inner_cv_mae=score,
            **{f'weight_{name}':float(w[list(subset).index(name)]) if name in subset else 0. for name in MODELS}))
    # Retrain each base model with outer-training labels only; evaluate on untouched outer fold.
    test_predictions={}
    for model in MODELS:
        test_predictions[model]=predict_model(model,outer_train,outer_test,outer_test,outer_fold,
            'outer',SEED+outer_fold*10000+9000)
    for subset_name,(subset,w) in selected.items():
        p=sum(float(x)*test_predictions[name] for name,x in zip(subset,w))
        for k,i in enumerate(outer_test):
            for j,iso in enumerate(ISOFORMS):
                if not np.isfinite(Y[i,j]):continue
                lo=CI_LOW[i,j];hi=CI_HIGH[i,j]
                excess=(max(lo-p[k,j],0)+max(p[k,j]-hi,0)) if np.isfinite(lo) and np.isfinite(hi) and lo<=hi else np.nan
                outer_records.append(dict(fold=outer_fold,subset=subset_name,
                    canonical_smiles=all_smiles[i],scaffold_group=all_groups[i],
                    isoform=iso,y_true=float(Y[i,j]),y_pred=float(p[k,j]),
                    ci_low=float(lo),ci_high=float(hi),interval_excess=float(excess)))
    print(f'Completed outer fold {outer_fold}; {len(outer_test)} compounds.')

selection=pd.DataFrame(selection_records)
outer_predictions=pd.DataFrame(outer_records)
inner_oof=pd.DataFrame(inner_records)
assert not outer_predictions.duplicated(['fold','subset','isoform','canonical_smiles']).any()
assert not inner_oof.duplicated(['fold','inner_fold','isoform','canonical_smiles']).any()
selection.to_csv(OUT/'nested_selected_weights.csv',index=False)
outer_predictions.to_csv(OUT/'nested_outer_predictions.csv',index=False)
inner_oof.to_csv(OUT/'inner_oof_predictions_for_selection.csv',index=False)
print('Outer records:',len(outer_predictions),'components:',len(SUBSETS),'folds:',sorted(outer_predictions.fold.unique()))
display(selection.round(4))

## 7 · Main ablation: which component contributes uniquely?

The strongest ablation removes a component from the three-model family, **re-tunes the remaining blend using the same inner OOF predictions**, and compares its outer-fold error to the triple. A positive drop-one value means the component was beneficial. We also report standalone model quality and isoform-specific gains. Because weights are selected globally, the isoform comparisons are descriptive, not separately optimized.

In [ ]:
def summarize(frame,by):
    rows=[]
    for keys,part in frame.groupby(by):
        keys=keys if isinstance(keys,tuple) else (keys,)
        stats=dict(zip(by,keys))
        yt=part.y_true.to_numpy(float);yp=part.y_pred.to_numpy(float)
        stats.update(n=len(part),mae=float(mean_absolute_error(yt,yp)),
            r2=float(r2_score(yt,yp)) if len(part)>1 else np.nan,
            interval_excess=float(part.interval_excess.mean()) if part.interval_excess.notna().any() else np.nan,
            prediction_sd_ratio=float(np.std(yp)/np.std(yt)) if np.std(yt)>1e-12 else np.nan)
        rows.append(stats)
    return pd.DataFrame(rows)

per_iso=summarize(outer_predictions,['subset','isoform'])
per_fold_iso=summarize(outer_predictions,['fold','subset','isoform'])
macro=per_iso.groupby('subset',as_index=False).agg(macro_mae=('mae','mean'),
   macro_interval_excess=('interval_excess','mean'),macro_r2=('r2','mean'))
macro=macro.sort_values('macro_mae')
full=blend_name(tuple(MODELS))
full_mae=float(macro.loc[macro.subset==full,'macro_mae'].iloc[0])
contribution_rows=[]
for model in MODELS:
    without=tuple(x for x in MODELS if x!=model)
    no_name=blend_name(without)
    no_mae=float(macro.loc[macro.subset==no_name,'macro_mae'].iloc[0])
    standalone=float(macro.loc[macro.subset==model,'macro_mae'].iloc[0])
    contribution_rows.append(dict(component=model,full_ensemble_mae=full_mae,
        without_component_mae=no_mae,drop_one_gain=no_mae-full_mae,
        standalone_mae=standalone))
contribution=pd.DataFrame(contribution_rows).sort_values('drop_one_gain',ascending=False)
per_cyp=[]
for iso in ISOFORMS:
    a=per_iso[per_iso.isoform==iso].set_index('subset')['mae']
    for model in MODELS:
        without=blend_name(tuple(x for x in MODELS if x!=model))
        per_cyp.append(dict(isoform=iso,component=model,
                            full_mae=float(a.loc[full]),without_mae=float(a.loc[without]),
                            drop_one_gain=float(a.loc[without]-a.loc[full]),
                            standalone_mae=float(a.loc[model])))
per_cyp=pd.DataFrame(per_cyp)

macro.to_csv(OUT/'ablation_macro_summary.csv',index=False)
per_iso.to_csv(OUT/'ablation_per_isoform.csv',index=False)
per_fold_iso.to_csv(OUT/'ablation_per_fold_isoform.csv',index=False)
contribution.to_csv(OUT/'drop_one_contribution.csv',index=False)
per_cyp.to_csv(OUT/'contribution_by_isoform.csv',index=False)
print('Macro ranking');display(macro.round(4))
print('Drop-one contributions (positive = model helps triple)');display(contribution.round(4))
print('By isoform');display(per_cyp.pivot(index='component',columns='isoform',values='drop_one_gain').round(4))

## 8 · Paired, scaffold-block uncertainty and visual interpretation

Bootstrap resampling of scaffold groups gives descriptive intervals **conditional on trained models, inner model selection, and these particular folds**. It is not a completely independent experimental validation or a substitute for repeated training seeds.

In [ ]:
def paired_cluster_bootstrap(frame,model_a,model_b,B=500,seed=SEED):
    a=frame[frame.subset==model_a][['canonical_smiles','isoform','scaffold_group','y_true','y_pred']].rename(columns={'y_pred':'pred_a'})
    b=frame[frame.subset==model_b][['canonical_smiles','isoform','y_pred']].rename(columns={'y_pred':'pred_b'})
    joint=a.merge(b,on=['canonical_smiles','isoform'],validate='one_to_one')
    assert len(joint)==len(a)==len(b)
    joint['difference']=np.abs(joint.y_true-joint.pred_a)-np.abs(joint.y_true-joint.pred_b)
    group_table=joint.groupby(['scaffold_group','isoform'],as_index=False).difference.agg(['sum','count']).reset_index()
    sums=group_table.pivot(index='scaffold_group',columns='isoform',values='sum').reindex(columns=ISOFORMS).fillna(0).to_numpy()
    counts=group_table.pivot(index='scaffold_group',columns='isoform',values='count').reindex(columns=ISOFORMS).fillna(0).to_numpy()
    rng=np.random.default_rng(seed)
    vals=[]
    for bidx in range(B):
        chosen=rng.integers(len(sums),size=len(sums))
        numerator=sums[chosen].sum(axis=0)
        denominator=counts[chosen].sum(axis=0)
        if (denominator==0).any():continue
        vals.append(float(np.mean(numerator/denominator)))
    values=np.asarray(vals,float)
    return dict(comparison=f'{model_a} -> {model_b}',
        improvement=float(np.mean([joint.loc[joint.isoform==iso,'difference'].mean() for iso in ISOFORMS])),
        bootstrap_low=float(np.quantile(values,.025)) if len(values) else np.nan,
        bootstrap_high=float(np.quantile(values,.975)) if len(values) else np.nan,
        bootstrap_n=len(values))

paired=[]
for model in MODELS:
    paired.append(paired_cluster_bootstrap(outer_predictions,model,full,B=200 if FAST_MODE else 1000))
paired=pd.DataFrame(paired)
paired.to_csv(OUT/'paired_scaffold_bootstrap.csv',index=False)
display(paired.round(4))

fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].barh(macro.subset[::-1],macro.macro_mae[::-1])
axes[0].set_xlabel('Macro MAE (lower is better)')
axes[0].set_title('Seven nested-selected model families')
pivot=per_cyp.pivot(index='component',columns='isoform',values='drop_one_gain').reindex(index=MODELS,columns=ISOFORMS)
im=axes[1].imshow(pivot.to_numpy(),cmap='coolwarm',aspect='auto')
axes[1].set_xticks(range(4),ISOFORMS,rotation=30,ha='right')
axes[1].set_yticks(range(3),MODELS)
axes[1].set_title('Drop-one MAE gain (positive = unique benefit)')
fig.colorbar(im,ax=axes[1],shrink=.7)
plt.tight_layout();plt.savefig(OUT/'ablation_overview.png',dpi=175);plt.show()

## 9 · Machine-readable audit and results ZIP

The ZIP includes outer predictions, nested inner OOF predictions, selected weights, fold manifests, per-CYP contributions, uncertainty summaries, and provenance. Training checkpoints and raw assay files stay outside the ZIP. **A smoke-test proxy output is flagged as unfit for scientific claims**.

In [ ]:
settings_file=OUT/'run_metadata.json'
complete=set(FOLDS_TO_RUN)==set(range(N_OUTER_FOLDS))
scientific=complete and (not FAST_MODE) and RUN_GRAPH and (not GRAPH_PROXY_SMOKE_ONLY)
metadata={**SETTINGS,'folds_run':FOLDS_TO_RUN,'complete_five_fold_oof':complete,
    'scientifically_valid_full_model_run':scientific,
    'n_primary':N_PRIMARY,'n_auxiliary_only':len(extra),
    'n_observed_direct':int(np.isfinite(Y).sum()),
    'n_screen_observed':int(np.isfinite(SCREEN_Y).sum()),
    'cache_signature':SIGNATURE,'cache_reused_calls':int(sum(x['cached'] for x in CACHED_CALLS)),
    'cache_total_calls':len(CACHED_CALLS),
    'metric_note':'MAE + mean interval excess are diagnostic; not official MA-ST-RAE',
    'selection_note':'Only inner OOF predictions select simplex blend weights; outer folds held out',
    'third_party_checkpoint_sha256':CKPT_SHA,
    'limitations':['This uses one five-fold partition, not repeated independent train seeds',
       'External pretrained foundation model data are not audited for molecular overlap',
       'Bootstrap CI conditions on fits and the selected analysis protocol']}
settings_file.write_text(json.dumps(metadata,indent=2,default=str))
pd.DataFrame(AUDIT_LOG).to_csv(OUT/'leakage_audit.csv',index=False)
pd.DataFrame(CACHED_CALLS).to_csv(OUT/'resume_audit.csv',index=False)
if AUDIT_LOG:
    assert (pd.DataFrame(AUDIT_LOG)[['outer_scaffold_overlap','validation_scaffold_overlap']]==0).all().all()
print('Complete outer folds?',complete,'Scientifically valid full model run?',scientific)
archive=Path(EXPORT_ZIP)
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):
        if path.is_file():z.write(path,OUT.name+'/'+path.name)
print('Saved:',archive, 'size KiB:',round(archive.stat().st_size/1024,1))

## 10 · Interpretation and reporting checklist

- [ ] `FAST_MODE=False`, all five outer folds completed, real CheMeleon model (not proxy).
- [ ] `leakage_audit.csv` shows zero scaffold overlap in all relevant inner/outer fits.
- [ ] Document `run_metadata.json`, primary dataset SHA, checkpoint SHA, and Chemprop version.
- [ ] Compare the triple with its three **nested-reselected drop-one** pairs, not just standalone models.
- [ ] Examine `contribution_by_isoform.csv` for negative transfer; a component may help one CYP and hurt another.
- [ ] Look at full-ensemble weights across outer folds in `nested_selected_weights.csv`. Highly variable weights suggest selection instability.
- [ ] Do not claim nested validation of the earlier 0.5404 retrospective blend; Notebook 11 is a separate, more expensive validation protocol.
- [ ] If results warrant a challenge submission, verify the organizers' **exact** MA-ST-RAE implementation before final model selection.

### What to send back

Upload `cyp_notebook_11_results.zip`. The most important outputs are `ablation_macro_summary.csv`, `drop_one_contribution.csv`, `contribution_by_isoform.csv`, `nested_selected_weights.csv`, and `leakage_audit.csv`.